# 05 Real Dataset Pipeline To Spec Roundtrip

`OctoSense.pipelines` and `OctoSense.specs` together form
OctoSense's `Benchmark Engine`. They solve the problem that a
wireless sensing experiment should be easy to run now, but also
easy to save, share, restore, and reproduce later. The design uses
pipelines for benchmark execution and `BenchmarkSpec` for benchmark
serialization, so one experiment can move between runnable workflow
and saved record without changing its meaning. This notebook shows
that result on the real notebook-local Widar3 sample under
`demo/notebook/data/widar3/` by building one pipeline, running it,
exporting YAML and JSON specs, loading them back, and rerunning the
restored pipeline on the same data splits.

### Set Up Pipeline And Spec Roundtrip Utilities

Import the public transform, serde, and timing helpers needed for a
real-data pipeline run followed by canonical spec export and reload.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = (NOTEBOOK_DIR / "../..").resolve()
SRC_ROOT = REPO_ROOT / "src"
TEST_DATA_ROOT = NOTEBOOK_DIR / "data"
PRESET_ROOT = NOTEBOOK_DIR / "presets"
NOTEBOOK_TREE_DEPTH = 2
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from loguru import logger
import octosense as octo
import time
import torch
from octosense.datasets.storage.downloader import download_dataset
from octosense.specs.serde.canonical import canonical_dump, spec_digest
from octosense.specs.serde.json_io import dump_json, load_json
from octosense.specs.serde.yaml_io import dump_yaml, load_yaml
from octosense.transforms.adapters.to_sequence import ToSequence
from octosense.transforms.modalities.wifi.sanitize import PilotSubcarrierInterpolate
from octosense.transforms.ops.axis import AxisNormalize
from octosense.transforms.ops.complex import Magnitude
from octosense.transforms.ops.normalize import Normalize
from octosense.transforms.ops.spectral import STFT
from octosense.transforms.ops.temporal import TemporalResize


logger.remove()
logger.add(sys.stderr, colorize=False, format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")
logger.info("Notebook setup completed with repository-relative paths")
logger.info("Default describe tree render depth set to {}", NOTEBOOK_TREE_DEPTH)

### Build A Real-Data Split Subset For A Runnable Notebook Pipeline

Reuse or download the notebook-local Widar3 sample, inspect the
public split views, and select a bounded subset per split so the
roundtrip stays runnable on CPU while preserving label coverage.

In [ ]:
logger.info("Preparing the real Widar3 sample root for the merged pipeline/spec walkthrough")
dataset_root = TEST_DATA_ROOT / "widar3"
canonical_sample_path = (
    dataset_root
    / "CSI"
    / "20181211"
    / "user3"
    / "user3-1-1-1-1-r1.dat"
)
if canonical_sample_path.exists():
    logger.info("Using existing notebook-local Widar3 sample root at {}", dataset_root.relative_to(REPO_ROOT))
else:
    logger.info("Downloading Widar3 sample payload into {}", dataset_root.relative_to(REPO_ROOT))
    dataset_root = download_dataset("widar3", root=dataset_root, part="sample", force=False)
    logger.info("Using downloaded Widar3 sample root at {}", dataset_root.relative_to(REPO_ROOT))

root_view = octo.datasets.load(
    "widar3",
    modalities=["wifi"],
    variant="sample",
    split_scheme="sample",
    task_binding="gesture",
    path=str(dataset_root),
)
available_splits = tuple(sorted({str(row["split"]) for row in root_view.metadata_rows()}))
assert {"train", "val", "test"}.issubset(set(available_splits))
split_limits = {"train": 24, "val": 12, "test": 12}
dataset = {}
for split_name, limit in split_limits.items():
    split_view = root_view.get_split(split_name)
    metadata_rows = split_view.metadata_rows()
    by_label = {}
    for index, row in enumerate(metadata_rows):
        by_label.setdefault(int(row["label"]), []).append(index)
    selected = [indices[0] for _, indices in sorted(by_label.items())]
    remaining = [
        index
        for _, indices in sorted(by_label.items())
        for index in indices[1:]
    ]
    selected.extend(remaining[: max(0, limit - len(selected))])
    dataset[split_name] = split_view.select(sorted(selected[:limit]))

split_counts = {split_name: len(view) for split_name, view in dataset.items()}
num_classes = len(dataset["train"].get_label_mapping())
print(root_view.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(dataset["train"].describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
logger.info("Available splits before subset selection: {}", available_splits)
logger.info("Selected real-data split counts: {}", split_counts)
logger.info("Label mapping: {}", dataset["train"].get_label_mapping())
logger.info("Using num_classes={} derived from the real train split", num_classes)

### Define The Transform Chain And Run The Original Pipeline

Assemble the public transform pipeline, bind it to the selected
dataset and model, and run infer/train/evaluate once so the notebook
captures the original runtime behavior before serialization.

In [ ]:
logger.info("Defining the explicit transform chain and selector-built model pipeline")
transform = octo.transforms.compose(
    [
        AxisNormalize(axis_name="subc", method="l2"),
        PilotSubcarrierInterpolate(axis_name="subc", method="linear"),
        TemporalResize(axis_name="time", target_length=256),
        STFT(axis_name="time", n_fft=256, hop_length=128, onesided=False),
        Magnitude(),
        Normalize(method="zscore", per_sample=True),
        ToSequence(
            flatten_axes=("subc", "tx", "rx", "freq"),
            time_axis="frame",
            value="magnitude",
        ),
    ]
)
pipeline = octo.pipelines.load(
    dataset=dataset,
    modalities=["wifi"],
    task_id="classification/gesture",
    task_binding="gesture",
    model_id="rfnet",
    entry_overrides={"num_classes": num_classes},
    transform=transform,
)
print(transform.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
print(pipeline.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
logger.info(
    "Transform operators: {}",
    [step.__class__.__name__ for step in transform.transforms],
)
logger.info(
    "Pipeline task/model: {} -> {}",
    pipeline.task_spec.task_id,
    pipeline.benchmark_spec.model.model_id,
)

infer_sample = dataset["test"][0][0]
quick_started = time.perf_counter()
original_logits = pipeline.infer(
    infer_sample,
    device="cpu",
    seed=7,
    batch_size=8,
    num_workers=0,
)
quick_elapsed = time.perf_counter() - quick_started
runtime_started = time.perf_counter()
original_train = pipeline.train(
    device="cpu",
    seed=7,
    batch_size=8,
    num_workers=0,
    epochs=1,
    progress=False,
)
runtime_elapsed = time.perf_counter() - runtime_started
original_eval = pipeline.evaluate(
    device="cpu",
    seed=7,
    batch_size=8,
    num_workers=0,
    progress=False,
)
exported_spec = pipeline.benchmark_spec
logger.info("Original train mode: {}", original_train["mode"])
logger.info("Original evaluate mode: {}", original_eval["mode"])
logger.info("Quick-path infer logits shape: {}", tuple(original_logits.shape))
logger.info("Quick-path infer wall time: {:.3f}s", quick_elapsed)
logger.info("Original run wall time: {:.3f}s", runtime_elapsed)
logger.info("Original primary metric: {}", original_eval["metrics"]["primary_metric_value"])
logger.info("Exported spec digest: {}", spec_digest(exported_spec))

### Serialize The BenchmarkSpec And Verify The Restored Pipeline

Write YAML and JSON spec artifacts, reload them through canonical
serde, rebuild the pipeline from the restored payload, and verify
that the roundtrip preserves the runnable behavior within tolerance.

In [ ]:
logger.info("Writing YAML and JSON spec artifacts, then restoring the canonical spec payload")
artifact_root = NOTEBOOK_DIR / "artifacts"
artifact_root.mkdir(parents=True, exist_ok=True)
spec_yaml_path = artifact_root / "roundtrip_spec.yaml"
spec_json_path = artifact_root / "roundtrip_spec.json"
spec_yaml_path.write_text(dump_yaml(exported_spec), encoding="utf-8")
spec_json_path.write_text(dump_json(exported_spec), encoding="utf-8")
yaml_loaded = load_yaml(spec_yaml_path)
json_loaded = load_json(spec_json_path)
logger.info("Dataclass equality after roundtrip: {}", exported_spec.__class__.from_dict(exported_spec.to_dict()) == exported_spec)
logger.info("YAML canonical payload matches exported spec: {}", canonical_dump(yaml_loaded) == canonical_dump(exported_spec))
logger.info("JSON canonical payload matches exported spec: {}", canonical_dump(json_loaded) == canonical_dump(exported_spec))
logger.info("Spec artifact root: {}", artifact_root.relative_to(REPO_ROOT))
logger.info("Exported transform operators: {}", [step.operator_id for step in exported_spec.transform.steps])

restored_pipeline = octo.pipelines.load(
    spec=json_loaded,
    dataset=dataset,
)
print(restored_pipeline.describe_tree().render(depth=NOTEBOOK_TREE_DEPTH))
restored_train = restored_pipeline.train(
    device="cpu",
    seed=7,
    batch_size=8,
    num_workers=0,
    epochs=1,
    progress=False,
)
restored_eval = restored_pipeline.evaluate(
    device="cpu",
    seed=7,
    batch_size=8,
    num_workers=0,
    progress=False,
)
restored_logits = restored_pipeline.infer(
    infer_sample,
    device="cpu",
    seed=7,
    batch_size=8,
    num_workers=0,
)
train_primary_close = torch.isclose(
    torch.tensor(float(original_train["metrics"]["primary_metric_value"])),
    torch.tensor(float(restored_train["metrics"]["primary_metric_value"])),
    atol=1e-6,
    rtol=1e-5,
).item()
eval_primary_close = torch.isclose(
    torch.tensor(float(original_eval["metrics"]["primary_metric_value"])),
    torch.tensor(float(restored_eval["metrics"]["primary_metric_value"])),
    atol=1e-6,
    rtol=1e-5,
).item()
logger.info("Restored train mode: {}", restored_train["mode"])
logger.info("Restored evaluate mode: {}", restored_eval["mode"])
logger.info(
    "Train primary metric stays within tolerance after spec roundtrip: {}",
    train_primary_close,
)
logger.info(
    "Evaluate primary metric stays within tolerance after spec roundtrip: {}",
    eval_primary_close,
)
logger.info(
    "Infer logits stay allclose after spec roundtrip: {}",
    torch.allclose(original_logits, restored_logits, atol=1e-6, rtol=1e-5),
)
logger.info("Restored primary metric: {}", restored_eval["metrics"]["primary_metric_value"])